In [1]:
import pandas as pd
import numpy as np
import pymysql
from sqlalchemy import create_engine
import getpass 
import mysql.connector


In [2]:
password = getpass.getpass()

In [3]:
bd = "sakila"
connection_string = 'mysql+pymysql://root:' + password + '@localhost/'+bd
engine = create_engine(connection_string)
engine.connect()

In [4]:
def rentals_month(engine, month, year):
    query = f"""
    SELECT *
    FROM rental
    WHERE MONTH(rental_date) = {month}
      AND YEAR(rental_date) = {year};
    """
    df = pd.read_sql(query, engine)
    return df

In [10]:
rentals_may = rentals_month(engine, 5, 2005)
rentals_may.head()

,rental_id,rental_date,inventory_id,customer_id,return_date,staff_id,last_update
0,1,2005-05-24 22:53:30,367,130,2005-05-26 22:04:30,1,2006-02-15 21:30:53
1,2,2005-05-24 22:54:33,1525,459,2005-05-28 19:40:33,1,2006-02-15 21:30:53
2,3,2005-05-24 23:03:39,1711,408,2005-06-01 22:12:39,1,2006-02-15 21:30:53
3,4,2005-05-24 23:04:41,2452,333,2005-06-03 01:43:41,2,2006-02-15 21:30:53
4,5,2005-05-24 23:05:21,2079,222,2005-06-02 04:33:21,1,2006-02-15 21:30:53


In [11]:
def rental_count_month(df, month, year):
    counts = df.groupby('customer_id')['rental_id'].count().reset_index()

    col_name = f"rentals_{month:02d}_{year}"
    counts.rename(columns={'rental_id': col_name}, inplace=True)
    
    return counts

In [12]:
rental_count_may = rental_count_month(rentals_may, 5, 2005)
rental_count_may.head()

,customer_id,rentals_05_2005
0,1,2
1,2,1
2,3,2
3,5,3
4,6,3


In [13]:
def compare_rentals(df1, df2):
    # merge on customer_id to keep only customers in both months
    df_combined = pd.merge(df1, df2, on='customer_id', how='inner')
    
    # get column names dynamically
    col1 = df1.columns[1]
    col2 = df2.columns[1]
    
    # create difference column
    df_combined['difference'] = df_combined[col2] - df_combined[col1]
    
    return df_combined

In [14]:
rentals_may = rentals_month(engine, 5, 2005)
rentals_june = rentals_month(engine, 6, 2005)

rental_count_may = rental_count_month(rentals_may, 5, 2005)
rental_count_june = rental_count_month(rentals_june, 6, 2005)

comparison = compare_rentals(rental_count_may, rental_count_june)
comparison.head()


,customer_id,rentals_05_2005,rentals_06_2005,difference
0,1,2,7,5
1,2,1,1,0
2,3,2,4,2
3,5,3,5,2
4,6,3,4,1
